# Bayern vs Leipzig (Bundesliga 2026/27, matchday 6): a pre-match forecast

Companion notebook to the [Jan's Matchday analysis](https://jhvl88.github.io/matchday/analysis/2026-27-md6-bayern-leipzig/).
It rebuilds the post from public data with the site's shared pipeline (`analysis/pipeline/`):

1. **Dixon–Coles** – time-weighted Poisson model with the low-score correction, learning from a 25% xG / 75% goals blend (JAX).
2. **Bayesian shots-to-goals model** – shots on target ➜ goals, dynamic team strengths, 1. + 2. Bundesliga, xG as a second view of chance creation; sampled with NUTS (NumPyro on JAX).
3. **XGBoost** – Poisson boosters on 36 leak-free pre-match features (Elo, form, xG form).
4. An equal-weight **ensemble**, then the checks in the post: the models against the bookmakers on big favourites and on goals, over 2,187 out-of-sample matches since 2019/20.

These are the **xG versions** of the models, adopted in October 2026 after a
[pre-registered test](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/xg/README.md).

**Runtime:** `Runtime ▸ Change runtime type ▸ GPU` (a T4 is enough). JAX runs the NUTS chains on the GPU and
XGBoost trains with CUDA. A full run takes about 10 minutes on a GPU, 10–15 on a CPU.

**Freeze:** only results up to matchday 5 (11 October 2026) are used, exactly as in the post. The source files are pinned
to the commit the post used. MCMC is random and GPUs add up in a different order, so expect numbers within about a
percentage point of the published ones.

In [ ]:
!nvidia-smi -L || echo "No GPU found - everything still runs on CPU, just slower"

In [ ]:
# ---------------------------------------------------------------- settings
REPO = "https://github.com/JHvL88/matchday"
SOURCE_COMMIT = "e47f4cf3f56300106803e42d3349de91c05943fb"    # source-data: football-data.co.uk files the post used
RESULTS_COMMIT = "7bcf73a1a114dd865178bd25a74e87cd37745314"   # backtest-results: walk-forward predictions (xg/)
POST = "matchday/analysis/2026-27-md6-bayern-leipzig"
PIPE = "matchday/analysis/pipeline"

import os, sys, json, subprocess
def sh(*args):
    subprocess.run(list(args), check=True)

if not os.path.exists("matchday"):
    sh("git", "clone", "-q", "--depth", "1", REPO, "matchday")
# keep Colab's own GPU build of JAX; add the rest
sh(sys.executable, "-m", "pip", "install", "-q", "numpyro==0.22.0", "xgboost")

def pinned(branch_dir, commit):
    # one commit of a branch, without its history
    if not os.path.exists(branch_dir):
        sh("git", "init", "-q", branch_dir)
        sh("git", "-C", branch_dir, "fetch", "-q", "--depth", "1", REPO, commit)
        sh("git", "-C", branch_dir, "checkout", "-q", "FETCH_HEAD")

pinned("source", SOURCE_COMMIT)
pinned("results", RESULTS_COMMIT)
import jax; print("JAX backend:", jax.default_backend())

## 1. Data

**Results, shots, shots on target, red cards and closing odds** come from football-data.co.uk, as stored (byte for byte, with
SHA-256 checksums) on the repo's `source-data` branch, pinned above. **xG** comes from Understat: one request per season since 2014/15.
Understat publishes no licence, so those files stay on your machine and only derived numbers are published.

In [ ]:
if not os.path.exists("understat/manifest.json"):
    if subprocess.run([sys.executable, f"{PIPE}/fetch_understat.py", "--out", "."]).returncode != 0:
        # Fallback for the site owner: the private copy, with a token in Colab's secrets (DATA_REPO_TOKEN)
        token = os.environ.get("DATA_REPO_TOKEN", "")
        try:
            from google.colab import userdata
            token = token or userdata.get("DATA_REPO_TOKEN")
        except Exception:
            pass
        auth = f"x-access-token:{token}@" if token else ""
        print("Understat not reachable: trying the private data repository instead.")
        sh("git", "clone", "-q", "--depth", "1", f"https://{auth}github.com/JHvL88/matchday-data", "matchday-data")
        sh("rm", "-rf", "understat"); sh("cp", "-r", "matchday-data/understat", "understat")
pub = {f["path"]: f["sha256"] for f in json.load(open("matchday/analysis/backtest/xg/understat_manifest.json"))["files"]}
mine = {f["path"]: f["sha256"] for f in json.load(open("understat/manifest.json"))["files"]}
print(sum(pub.get(p) == mine[p] for p in mine), "of", len(mine), "Understat files byte-identical to the published xG run")
print("Changed since (fine for finished seasons only if Understat revised them):", sorted(p for p in mine if p in pub and pub[p] != mine[p]) or "none")

sh(sys.executable, f"{PIPE}/build_matches.py", "source", "matches.csv", "--understat", "understat")

## 2. Fit the three models and freeze the forecast

`post_model.py` imports the model set-up from `analysis/pipeline/forecast.py`, the same code the backtest graded:
Dixon–Coles on eight seasons with a one-year half-life, XGBoost trained on earlier seasons only, the Bayesian model on six
seasons plus this one (two NUTS chains, 800 warm-up + 800 draws each). The ensemble is the plain average of the three
score grids. Equal weights were fixed before any backtest, and they stay fixed.

In [ ]:
sh(sys.executable, f"{POST}/post_model.py", "matches.csv", "out")
f = json.load(open("out/forecast.json"))
print(json.dumps(f["config"], indent=1))
e = f["fixtures"][0]["models"]["ens"]
print(f"Ensemble: Bayern {e['H']:.1%}, draw {e['D']:.1%}, Leipzig {e['A']:.1%}; expected goals {e['lh']:.2f}-{e['la']:.2f}")

The ensemble in three lines (from `post_model.py`):

```python
lh, la = dc.rates("Bayern Munich", "RB Leipzig")
grids = dict(dc=score_matrix(lh, la, dc.rho), bay=B, xgb=score_matrix(xlh, xla, dc.rho))
grids["ens"] = (grids["dc"] + grids["bay"] + grids["xgb"]) / 3     # pre-registered equal weights
```

## 3. Compare with the published forecast

In [ ]:
pub = json.load(open(f"{POST}/forecast.json"))["fixtures"][0]["models"]
mine = f["fixtures"][0]["models"]
for k, lab in [("dc", "Dixon-Coles"), ("bay", "Bayesian"), ("xgb", "XGBoost"), ("ens", "Ensemble")]:
    print(f"{lab:12s} published {pub[k]['H']:.3f} / {pub[k]['D']:.3f} / {pub[k]['A']:.3f}"
          f"   this run {mine[k]['H']:.3f} / {mine[k]['D']:.3f} / {mine[k]['A']:.3f}")

## 4. The checks and the charts

`post_charts.py` loads the walk-forward predictions of all three models for every Bundesliga match since 2019/20
(the `backtest-results` branch, pinned above; to regenerate them see
[`analysis/backtest/backtest.ipynb`](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/backtest.ipynb)) and asks two questions:

* **Big favourites.** Group matches by the bookmakers' probability for their favourite. Compare what the models said, what the
  bookmakers said and how often the favourite won.
* **Goals.** Compare the models' expected total goals with the goals actually scored, overall and in Bayern's matches.

The bookmakers' prices for this match (`market.json`) were recorded from The Odds API by the site's odds recorder; only
derived figures are published. Then every chart in the post is drawn, in English and German.

In [ ]:
import shutil
os.makedirs("post", exist_ok=True)
shutil.copy(f"{POST}/market.json", "post/market.json")
sh(sys.executable, f"{POST}/post_charts.py", "matches.csv", "results/xg", "out", "post", "source")
n = json.load(open("post/numbers.json"))
for row, lab in zip(n["favourites"], ["fav 50-60%", "fav 60-70%", "fav 70-80%", "fav 80%+", "Bayern at home"]):
    print(f"{lab:15s} n={row['n']:4d}  models {row['models']:.1%}  bookmakers {row['market']:.1%}  happened {row['actual']:.1%}")

The favourite check, in full:

```python
fav_home = o.avgc_H >= o.avgc_A                        # the bookmakers' favourite
o["p_mkt"] = np.where(fav_home, o.avgc_H, o.avgc_A)    # their probability for it (margin removed)
o["p_ens"] = np.where(fav_home, o.ens_H, o.ens_A)      # the ensemble's probability for the same team
o["won"] = np.where(fav_home, o.y == 0, o.y == 2)      # did it win?
for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    k = (o.p_mkt >= lo) & (o.p_mkt < hi)
    print(lo, k.sum(), o.p_ens[k].mean(), o.p_mkt[k].mean(), o.won[k].mean())
```

In [ ]:
from IPython.display import SVG, display
import glob
for path in sorted(glob.glob("post/figs/*.svg")):
    print(path); display(SVG(path))